<a href="https://colab.research.google.com/github/awaisfarooqchaudhry/IB9AU-GenerativeAI-2026/blob/main/Task13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 13 — Synthetic Fraud Transaction Data with CTGAN & TVAE (Google Colab)

This notebook is a **complete beginner-friendly Colab notebook** for **Required Task 13**.

It is adapted from your uploaded reference notebook **`SD4_Synthetic_Loan_Data_Generation_with_CTGAN_&_TVAE.ipynb`**, but rewritten for the actual assignment task:

- load **`fraud_transactions.csv`**
- create a **synthetic dataset of 5000 records**
- **evaluate the quality** of the synthetic data

## What this notebook does
1. upload `fraud_transactions.csv`
2. inspect and lightly clean the data
3. detect likely ID-like columns and optionally remove them from modeling
4. auto-detect SDV metadata
5. train **CTGAN** and **TVAE**
6. sample **5000 synthetic rows** from each model
7. evaluate them using:
   - **SDV diagnostic report**
   - **SDV quality report**
   - simple **distribution comparisons**
   - optional **fraud-model utility test** if a target column is detected
8. choose the better model and save the final synthetic dataset

## Colab recommendation
Use **Runtime → Change runtime type → T4 GPU**.

## Beginner note
Run the notebook **from top to bottom**.
Do not skip cells.


## Step 0 — Install packages

This notebook uses the current **SDV** workflow:
- `CTGANSynthesizer`
- `TVAESynthesizer`
- `run_diagnostic`
- `evaluate_quality`

Run this cell first.


In [ ]:
!pip -q install -U sdv scikit-learn matplotlib seaborn

## Step 1 — Imports and setup


In [ ]:
import os
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
from sdv.evaluation.single_table import run_diagnostic, evaluate_quality, get_column_plot

warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("task13_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 200)
print("✅ Imports ready.")

## Step 2 — Upload `fraud_transactions.csv`

Upload the file when prompted.


In [ ]:
from google.colab import files

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file uploaded. Please upload fraud_transactions.csv")

uploaded_name = list(uploaded.keys())[0]
csv_path = f"/content/{uploaded_name}"

print("✅ Uploaded file:", csv_path)

## Step 3 — Load the CSV


In [ ]:
df_raw = pd.read_csv(csv_path)

print("Shape:", df_raw.shape)
display(df_raw.head())
print("\nData types:")
display(df_raw.dtypes)

## Step 4 — Basic cleaning and type handling

This cell:
- strips spaces from column names
- removes completely empty columns
- removes constant columns
- tries to parse obvious date/time columns
- converts object columns to clean strings

This is intentionally light-touch because SDV can handle mixed tabular data.


In [ ]:
df = df_raw.copy()
df.columns = [str(c).strip() for c in df.columns]

# Drop all-empty columns
all_empty_cols = [c for c in df.columns if df[c].isna().all()]
if all_empty_cols:
    df = df.drop(columns=all_empty_cols)

# Drop constant columns
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
if constant_cols:
    df = df.drop(columns=constant_cols)

def maybe_parse_datetime(series, col_name):
    if series.dtype != "object":
        return series

    lower_name = str(col_name).lower()
    likely_datetime_name = any(token in lower_name for token in ["date", "time", "timestamp", "datetime"])

    sample = series.dropna().astype(str).head(500)
    if len(sample) == 0:
        return series

    parsed = pd.to_datetime(sample, errors="coerce", utc=False)
    success_rate = parsed.notna().mean()

    if likely_datetime_name and success_rate >= 0.7:
        return pd.to_datetime(series, errors="coerce", utc=False)

    return series

for col in df.columns:
    df[col] = maybe_parse_datetime(df[col], col)

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype(str).replace({"nan": np.nan}).str.strip()

print("✅ Cleaned shape:", df.shape)
print("Dropped all-empty columns:", all_empty_cols)
print("Dropped constant columns:", constant_cols)
display(df.head())

## Step 5 — Quick data audit

This helps you understand what kind of data you have before training the synthesizers.


In [ ]:
summary_df = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    "nunique": [df[c].nunique(dropna=True) for c in df.columns],
})

display(summary_df)

print("\nRows:", len(df))
print("Columns:", len(df.columns))

## Step 6 — Detect likely ID-like columns

ID-like columns often hurt tabular synthesis quality because they are nearly unique and
do not represent useful patterns.

Examples:
- transaction IDs
- row IDs
- hashes
- reference numbers

This cell flags such columns. By default, we remove them **from modeling only**.


In [ ]:
def detect_id_like_columns(dataframe):
    candidates = []
    for col in dataframe.columns:
        col_lower = col.lower()
        nunique = dataframe[col].nunique(dropna=True)
        ratio = nunique / max(len(dataframe), 1)

        name_hint = any(token in col_lower for token in ["id", "transaction_id", "txn", "reference", "ref", "uuid", "hash"])
        near_unique = ratio > 0.98 and nunique > 50

        if name_hint or near_unique:
            candidates.append(col)
    return sorted(list(set(candidates)))

id_like_cols = detect_id_like_columns(df)
print("Potential ID-like columns:", id_like_cols)

## Step 7 — Create the modeling dataframe

By default, the notebook removes ID-like columns from the synthesizer training data.

If you want to keep them, set `DROP_ID_LIKE_COLUMNS = False`.


In [ ]:
DROP_ID_LIKE_COLUMNS = True

if DROP_ID_LIKE_COLUMNS and len(id_like_cols) > 0:
    df_model = df.drop(columns=id_like_cols).copy()
else:
    df_model = df.copy()

print("Modeling shape:", df_model.shape)
display(df_model.head())

## Step 8 — Optional speed control

If your dataset is very large, you can train on a subset for faster Colab runs.

- Set `MAX_TRAIN_ROWS = None` to use the full dataset
- Set `MAX_TRAIN_ROWS = 20000` (for example) to speed things up

The synthetic output will still contain **5000 rows**.


In [ ]:
MAX_TRAIN_ROWS = None  # Example faster option: 20000

if MAX_TRAIN_ROWS is not None and len(df_model) > MAX_TRAIN_ROWS:
    df_train_model = df_model.sample(n=MAX_TRAIN_ROWS, random_state=42).reset_index(drop=True)
else:
    df_train_model = df_model.reset_index(drop=True)

print("Rows used for synthesizer training:", len(df_train_model))

## Step 9 — Detect metadata automatically

According to the SDV docs, `detect_from_dataframe(...)` is the standard way to create metadata from a pandas DataFrame.


In [ ]:
metadata = Metadata.detect_from_dataframe(
    data=df_train_model,
    table_name="fraud_transactions"
)

metadata.validate()
metadata.save_to_json(str(OUTPUT_DIR / "metadata.json"), mode="write")

print("✅ Metadata detected and validated.")
metadata_dict = metadata.to_dict()
print("Metadata keys:", metadata_dict.keys())
print("\nFirst few metadata column entries:")
display(pd.DataFrame(metadata_dict["tables"]["fraud_transactions"]["columns"]).T.head(10))

## Step 10 — Set synthesizer hyperparameters

These values are chosen to be more **Colab-friendly** than very large defaults.

You can increase the epochs later if you want better quality and are willing to wait longer.


In [ ]:
SYNTH_ROWS = 5000

CTGAN_EPOCHS = 30
TVAE_EPOCHS = 30

ctgan = CTGANSynthesizer(
    metadata=metadata,
    epochs=CTGAN_EPOCHS,
    enforce_min_max_values=True,
    enforce_rounding=True,
    verbose=True
)

tvae = TVAESynthesizer(
    metadata=metadata,
    epochs=TVAE_EPOCHS,
    enforce_min_max_values=True,
    enforce_rounding=True,
    verbose=True
)

print("✅ CTGAN and TVAE initialized.")

## Step 11 — Train CTGAN


In [ ]:
ctgan.fit(df_train_model)
ctgan.save(str(OUTPUT_DIR / "ctgan_synthesizer.pkl"))
print("✅ CTGAN training finished.")

## Step 12 — Train TVAE


In [ ]:
tvae.fit(df_train_model)
tvae.save(str(OUTPUT_DIR / "tvae_synthesizer.pkl"))
print("✅ TVAE training finished.")

## Step 13 — Generate 5000 synthetic rows from both models


In [ ]:
synthetic_ctgan = ctgan.sample(num_rows=SYNTH_ROWS)
synthetic_tvae = tvae.sample(num_rows=SYNTH_ROWS)

synthetic_ctgan.to_csv(OUTPUT_DIR / "synthetic_ctgan_5000.csv", index=False)
synthetic_tvae.to_csv(OUTPUT_DIR / "synthetic_tvae_5000.csv", index=False)

print("CTGAN synthetic shape:", synthetic_ctgan.shape)
print("TVAE synthetic shape :", synthetic_tvae.shape)

display(synthetic_ctgan.head())
display(synthetic_tvae.head())

## Step 14 — Evaluate CTGAN with SDV diagnostic and quality reports

The SDV docs recommend using:
- `run_diagnostic(...)` for validity checks
- `evaluate_quality(...)` for statistical similarity


In [ ]:
ctgan_diagnostic = run_diagnostic(
    real_data=df_train_model,
    synthetic_data=synthetic_ctgan,
    metadata=metadata,
    verbose=True
)

ctgan_quality = evaluate_quality(
    real_data=df_train_model,
    synthetic_data=synthetic_ctgan,
    metadata=metadata,
    verbose=True
)

ctgan_diag_score = float(ctgan_diagnostic.get_score())
ctgan_quality_score = float(ctgan_quality.get_score())

print("CTGAN diagnostic score:", round(ctgan_diag_score, 4))
print("CTGAN quality score   :", round(ctgan_quality_score, 4))

## Step 15 — Evaluate TVAE with SDV diagnostic and quality reports


In [ ]:
tvae_diagnostic = run_diagnostic(
    real_data=df_train_model,
    synthetic_data=synthetic_tvae,
    metadata=metadata,
    verbose=True
)

tvae_quality = evaluate_quality(
    real_data=df_train_model,
    synthetic_data=synthetic_tvae,
    metadata=metadata,
    verbose=True
)

tvae_diag_score = float(tvae_diagnostic.get_score())
tvae_quality_score = float(tvae_quality.get_score())

print("TVAE diagnostic score:", round(tvae_diag_score, 4))
print("TVAE quality score   :", round(tvae_quality_score, 4))

## Step 16 — Compare the two models

Higher scores are generally better:
- **Diagnostic score** checks validity / structural correctness
- **Quality score** checks statistical similarity


In [ ]:
comparison_df = pd.DataFrame({
    "model": ["CTGAN", "TVAE"],
    "diagnostic_score": [ctgan_diag_score, tvae_diag_score],
    "quality_score": [ctgan_quality_score, tvae_quality_score]
}).sort_values(by=["quality_score", "diagnostic_score"], ascending=False).reset_index(drop=True)

display(comparison_df)

## Step 17 — Choose the better model and mark the final synthetic dataset

This notebook picks the model with the **higher quality score**.
If the quality score ties, the diagnostic score breaks the tie.


In [ ]:
best_model_name = comparison_df.loc[0, "model"]

if best_model_name == "CTGAN":
    final_synthetic = synthetic_ctgan.copy()
    final_quality = ctgan_quality
    final_diagnostic = ctgan_diagnostic
else:
    final_synthetic = synthetic_tvae.copy()
    final_quality = tvae_quality
    final_diagnostic = tvae_diagnostic

final_synthetic_path = OUTPUT_DIR / "final_synthetic_fraud_transactions_5000.csv"
final_synthetic.to_csv(final_synthetic_path, index=False)

print("✅ Chosen final model:", best_model_name)
print("✅ Final synthetic dataset saved to:", final_synthetic_path)

## Step 18 — Look inside the quality report details

The quality report includes:
- **Column Shapes**
- **Column Pair Trends**

These help explain which columns were easy or hard to synthesize.


In [ ]:
column_shapes_details = final_quality.get_details("Column Shapes")
column_pair_details = final_quality.get_details("Column Pair Trends")

print("Top 10 weakest column shape scores:")
display(column_shapes_details.sort_values(by="Score", ascending=True).head(10))

print("Top 10 weakest column pair trend scores:")
display(column_pair_details.sort_values(by="Score", ascending=True).head(10))

## Step 19 — Simple distribution comparison for a few columns

This plots a few columns side by side to give you a visual feel for real vs synthetic.


In [ ]:
def choose_plot_columns(dataframe, max_plots=4):
    numeric_cols = dataframe.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = dataframe.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    chosen = []
    chosen.extend(numeric_cols[:2])
    chosen.extend(categorical_cols[:2])

    # Fill if fewer than requested
    for c in dataframe.columns:
        if c not in chosen:
            chosen.append(c)
        if len(chosen) >= max_plots:
            break

    return chosen[:max_plots]

plot_columns = choose_plot_columns(df_train_model, max_plots=4)
print("Columns selected for visual comparison:", plot_columns)

for col in plot_columns:
    try:
        fig = get_column_plot(
            real_data=df_train_model,
            synthetic_data=final_synthetic,
            metadata=metadata,
            column_name=col
        )
        fig.show()
    except Exception as e:
        print(f"Could not plot {col}: {e}")

## Step 20 — Optional utility test

If the dataset contains a likely fraud target column, this section checks whether a model
trained on synthetic data can still predict the target on real data.

This is a useful downstream quality test.

### Interpretation
- **TRTR** = Train on Real, Test on Real
- **TSTR** = Train on Synthetic, Test on Real


In [ ]:
def detect_target_column(dataframe):
    candidates = []
    exact_priority = [
        "is_fraud", "fraud", "fraud_flag", "fraudulent", "label",
        "class", "target", "isfraud", "y"
    ]

    cols_lower = {c.lower(): c for c in dataframe.columns}

    for name in exact_priority:
        if name in cols_lower:
            col = cols_lower[name]
            if dataframe[col].nunique(dropna=True) <= 10:
                return col

    for col in dataframe.columns:
        low = col.lower()
        if ("fraud" in low or "label" in low or "target" in low or low == "class") and dataframe[col].nunique(dropna=True) <= 10:
            return col

    return None

target_col = detect_target_column(df_train_model)
print("Detected target column:", target_col)

In [ ]:
def prepare_tabular_for_model(train_df, test_df, target_col):
    X_train = train_df.drop(columns=[target_col]).copy()
    X_test = test_df.drop(columns=[target_col]).copy()
    y_train = train_df[target_col].copy()
    y_test = test_df[target_col].copy()

    # Convert datetimes to integers for sklearn
    for frame in [X_train, X_test]:
        dt_cols = frame.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()
        for c in dt_cols:
            frame[c] = pd.to_datetime(frame[c], errors="coerce").astype("int64") // 10**9
            frame[c] = frame[c].replace(-9223372037, np.nan)

    combined = pd.concat([X_train, X_test], axis=0)
    combined = pd.get_dummies(combined, dummy_na=True)

    X_train_enc = combined.iloc[:len(X_train)].copy()
    X_test_enc = combined.iloc[len(X_train):].copy()

    X_train_enc = X_train_enc.fillna(0)
    X_test_enc = X_test_enc.fillna(0)

    return X_train_enc, X_test_enc, y_train, y_test


utility_results = None

if target_col is not None:
    target_nunique = df_train_model[target_col].nunique(dropna=True)
    if target_nunique <= 10:
        real_train, real_test = train_test_split(
            df_train_model,
            test_size=0.25,
            random_state=42,
            stratify=df_train_model[target_col] if target_nunique > 1 else None
        )

        # Baseline: Train on real, test on real
        X_real_train, X_real_test, y_real_train, y_real_test = prepare_tabular_for_model(real_train, real_test, target_col)
        rf_real = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
        rf_real.fit(X_real_train, y_real_train)
        pred_real = rf_real.predict(X_real_test)

        baseline = {
            "model": "TRTR (real->real)",
            "accuracy": accuracy_score(y_real_test, pred_real),
            "f1": f1_score(y_real_test, pred_real, average="weighted")
        }

        if target_nunique == 2:
            proba_real = rf_real.predict_proba(X_real_test)[:, 1]
            baseline["roc_auc"] = roc_auc_score(y_real_test, proba_real)
            baseline["avg_precision"] = average_precision_score(y_real_test, proba_real)

        # Synthetic utility from final chosen model
        synth_for_train = final_synthetic.copy()

        # Make sure target column exists
        if target_col in synth_for_train.columns:
            X_syn_train, X_syn_test, y_syn_train, y_syn_test = prepare_tabular_for_model(synth_for_train, real_test, target_col)
            rf_syn = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
            rf_syn.fit(X_syn_train, y_syn_train)
            pred_syn = rf_syn.predict(X_syn_test)

            tstr = {
                "model": f"TSTR ({best_model_name}->real)",
                "accuracy": accuracy_score(y_syn_test, pred_syn),
                "f1": f1_score(y_syn_test, pred_syn, average="weighted")
            }

            if target_nunique == 2:
                proba_syn = rf_syn.predict_proba(X_syn_test)[:, 1]
                tstr["roc_auc"] = roc_auc_score(y_syn_test, proba_syn)
                tstr["avg_precision"] = average_precision_score(y_syn_test, proba_syn)

            utility_results = pd.DataFrame([baseline, tstr])

if utility_results is not None:
    display(utility_results)
else:
    print("Utility evaluation skipped because no suitable target column was detected.")

## Step 21 — Save key report outputs


In [ ]:
comparison_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
column_shapes_details.to_csv(OUTPUT_DIR / "quality_column_shapes_details.csv", index=False)
column_pair_details.to_csv(OUTPUT_DIR / "quality_column_pair_details.csv", index=False)

if utility_results is not None:
    utility_results.to_csv(OUTPUT_DIR / "utility_results.csv", index=False)

summary = {
    "rows_in_original_data": int(len(df)),
    "rows_used_for_training": int(len(df_train_model)),
    "columns_used_for_modeling": list(df_train_model.columns),
    "dropped_all_empty_columns": all_empty_cols,
    "dropped_constant_columns": constant_cols,
    "dropped_id_like_columns": id_like_cols if DROP_ID_LIKE_COLUMNS else [],
    "ctgan_diagnostic_score": ctgan_diag_score,
    "ctgan_quality_score": ctgan_quality_score,
    "tvae_diagnostic_score": tvae_diag_score,
    "tvae_quality_score": tvae_quality_score,
    "chosen_final_model": best_model_name,
    "final_synthetic_rows": int(len(final_synthetic)),
    "target_column_detected": target_col
}

with open(OUTPUT_DIR / "task13_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print("✅ Saved summary and report tables.")

## Step 22 — Print a short final interpretation

This gives you text you can adapt for your submission.


In [ ]:
print("===== Task 13 Final Interpretation =====")
print(f"Original data shape: {df.shape}")
print(f"Training data used for synthesis: {df_train_model.shape}")
print(f"CTGAN quality score: {ctgan_quality_score:.4f}")
print(f"TVAE quality score : {tvae_quality_score:.4f}")
print(f"Chosen final model : {best_model_name}")
print(f"Final synthetic dataset shape: {final_synthetic.shape}")

if utility_results is not None:
    print("\nUtility comparison:")
    display(utility_results)

print("\nSuggested interpretation:")
print(
    f"The notebook generated 5000 synthetic fraud-transaction records using both CTGAN and TVAE, "
    f"evaluated both models using SDV diagnostic and quality reports, and selected {best_model_name} "
    f"as the final synthesizer because it achieved the stronger evaluation score on this dataset. "
    f"The final synthetic dataset has been saved for submission and further analysis."
)

## Step 23 — Download the main output files


In [ ]:
from google.colab import files

files.download(str(final_synthetic_path))
files.download(str(OUTPUT_DIR / "model_comparison.csv"))
files.download(str(OUTPUT_DIR / "task13_summary.json"))

## Step 24 — Optional: zip the whole outputs folder


In [ ]:
import shutil

archive_path = shutil.make_archive("task13_outputs_archive", "zip", OUTPUT_DIR)
print("✅ Created archive:", archive_path)

from google.colab import files
files.download(archive_path)

## Troubleshooting

### If training is too slow
Reduce the work by changing:
```python
MAX_TRAIN_ROWS = 20000
CTGAN_EPOCHS = 15
TVAE_EPOCHS = 15
```

### If the CSV has a weird separator
Replace:
```python
pd.read_csv(csv_path)
```
with something like:
```python
pd.read_csv(csv_path, sep=';')
```

### If the model fails on an ID column
Leave `DROP_ID_LIKE_COLUMNS = True`

### If the quality score is not very high
That is normal for difficult fraud data, especially if:
- the fraud class is rare
- there are many categorical columns
- there are many near-unique values
- the dataset is very small or very noisy

In that case, increase the epochs and try again.
